# 04 — ModelOpt Q/DQ ONNX Export
Change `QUANT_DTYPE` to `'int8'`, `'fp8'`, or `'int4'` and run all.

In [4]:
from pathlib import Path
import sys

sys.path.insert(0, str(Path('../src').resolve()))

In [5]:
import numpy as np
from data import build_runner_loaders
from config import ExperimentConfig
import modelopt.onnx.quantization as moq
import modelopt.torch.quantization as mtq

In [6]:
from model import get_model
from onnx_exporter import ONNXExporter

model = get_model(cfg=None, pretrained=False)

exporter = ONNXExporter(
    model=model,
    device="cpu",
    onnx_path="../onnx/resnet18.onnx",
)
exporter.export_model(opset_version=17, dynamic_batch=True)

/home/pf4636/code/resnet/quantized_resnets/src/onnx_exporter.py:26: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0430 10:47:09.128000 149519 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[onnx_exporter] Exporting to ../onnx/resnet18.onnx ...


W0430 10:47:09.523000 149519 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0430 10:47:09.524000 149519 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0430 10:47:09.525000 149519 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0430 10:47:09.526000 149519 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.


[torch.onnx] Obtain model graph for `ResNet18([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ResNet18([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).
Failed to convert the model to the target version 17 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "/home/pf4636/code/resnet/quantized_resnets/.venv/lib/python3.12/site-packages/onnxscript/version_converter/__init__.py", line 120, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/pf4636/code/resnet/quantized_resnets/.venv/lib/python3.12/site-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/home/pf4636/code/res

[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 41 of general pattern rewrite rules.
[onnx_exporter] Saved (0.1 MB)


PosixPath('../onnx/resnet18.onnx')

In [7]:
QUANT_DTYPE    = "fp8"        #fp8, int4, int8  
ONNX_IN        = "../onnx/resnet18.onnx"
ONNX_OUT       = f"../onnx/resnet18_{QUANT_DTYPE}_qdq.onnx"
CALIB_BATCHES  = 32

In [8]:
# Build calibration numpy array from dataloader
cfg = ExperimentConfig(device="cpu", batch_size=32)
loader, _ = build_runner_loaders(cfg)

batches = []
for i, (images, _) in enumerate(loader):
    if i >= CALIB_BATCHES:
        break
    batches.append(images.numpy())

calib_data = np.concatenate(batches, axis=0)
print(f"Calibration data: {calib_data.shape}")

[data] Train: 124,395  Holdout-Val: 5,000  (num_classes=100, val_per_class=50, seed=42)
Calibration data: (1024, 3, 224, 224)


moq.quantize(
        onnx_path=ONNX_IN,
        quantize_mode=QUANT_DTYPE,
        calibration_data=calib_data,
        output_path=ONNX_OUT,
        op_types_to_quantize=["Conv", "MatMul", "Gemm", "Add"]
        
    )
print(f"Saved → {ONNX_OUT}")

In [9]:
import onnx as _onnx
_FP8_OPS = {"Conv", "Gemm", "MatMul", "Add"}
_nodes_to_q = None
if QUANT_DTYPE == "fp8":
    _m = _onnx.load(ONNX_IN)
    _nodes_to_q = [n.name for n in _m.graph.node if n.op_type in _FP8_OPS]
    print(f"Explicit nodes_to_quantize: {len(_nodes_to_q)} nodes "
          f"({sum(1 for n in _m.graph.node if n.op_type == 'Add')} Add nodes included)")

moq.quantize(
    onnx_path=ONNX_IN,
    quantize_mode=QUANT_DTYPE,
    calibration_data=calib_data,
    output_path=ONNX_OUT,
    op_types_to_quantize=list(_FP8_OPS) if QUANT_DTYPE == "fp8" else None,
    nodes_to_quantize=_nodes_to_q,
)
print(f"Saved → {ONNX_OUT}")

Explicit nodes_to_quantize: 29 nodes (8 Add nodes included)
[modelopt][onnx] - INFO - Starting quantization process for model: ../onnx/resnet18.onnx
[modelopt][onnx] - INFO - Quantization mode: fp8
[modelopt][onnx] - INFO - Preprocessing the model ../onnx/resnet18.onnx
[modelopt][onnx] - INFO - Model has dynamic inputs: ['images']
[modelopt][onnx] - INFO - Found 0 custom layers and 59 tensors
[modelopt][onnx] - INFO - No custom ops found. If that's not correct, please make sure that the 'tensorrt' python package is correctly installed and that the paths to 'libcudnn*.so' and TensorRT 'lib/' are in 'LD_LIBRARY_PATH'. If the custom op is not directly available as a plugin in TensorRT, please also make sure that the path to the compiled '.so' TensorRT plugin is also being given via the  '--trt_plugins' flag (requires TRT 10+).
[modelopt][onnx] - INFO - Duplicating shared constants
[modelopt][onnx] - INFO - Setting up CalibrationDataProvider for calibration
[modelopt][onnx] - INFO - Analyz

100%|██████████| 43/43 [00:00<00:00, 2063.94it/s]


100%|██████████| 43/43 [00:00<00:00, 2313.46it/s]


100%|██████████| 43/43 [00:00<00:00, 2389.28it/s]


100%|██████████| 43/43 [00:00<00:00, 2433.74it/s]


...

100%|██████████| 43/43 [00:00<00:00, 1809.05it/s]


100%|██████████| 43/43 [00:00<00:00, 2254.07it/s]


Finding optimal threshold for each tensor using 'entropy' algorithm ...
Number of tensors : 43
Number of histogram bins : 128 (The number may increase depends on the data it collects)
Number of quantized bins : 128
[modelopt][onnx] - INFO - Starting post-processing of quantized model
[modelopt][onnx] - INFO - Deleting QDQ nodes from marked inputs to make certain operations fusible
[modelopt][onnx] - INFO - Converting float tensors to fp16


Found QuantizeLinear/DequantizeLinear nodes. Updating minimum opset from 13 to 19.
Shared constants were detected and duplicated accordingly.
Some initializers contain values smaller than smallest fp16 value, values will be replaced with 6.0e-08.


[modelopt][onnx] - INFO - Starting INT8 to FP8 conversion
[modelopt][onnx] - INFO - FP8 quantization completed in 37.21 seconds
[W] colored module is not installed, will not use colors when logging. To enable colors, please install the colored module: python3 -m pip install colored
[W] Could not convert: FLOAT8E4M3FN to a corresponding NumPy type. The original ONNX type will be preserved. 
[modelopt][onnx] - INFO - Total number of nodes: 145
[modelopt][onnx] - INFO - Total number of quantized nodes: 27
[modelopt][onnx] - INFO - Quantized onnx model is saved as ../onnx/resnet18_fp8_qdq.onnx
[modelopt][onnx] - INFO - Cleaning up intermediate files
[modelopt][onnx] - INFO - Validating quantized model
[modelopt][onnx] - INFO - Quantization process completed
Saved → ../onnx/resnet18_fp8_qdq.onnx


In [10]:
import onnx
ops = [n.op_type for n in onnx.load(f"../onnx/resnet18_{QUANT_DTYPE}_qdq.onnx").graph.node]
print(f"Q: {ops.count('QuantizeLinear')}, DQ: {ops.count('DequantizeLinear')}")

Q: 46, DQ: 46


For Int4:

This is the key insight you're missing: INT4 almost always means weight-only quantization, which is a fundamentally different scheme.
Here's why. INT4 has only 16 discrete values (or 8 for unsigned). That's far too coarse for activations, which vary dynamically at inference time. So instead of the full Q/DQ pattern, INT4 tools do this:

Weights are quantized offline and stored as INT4 constants directly in the graph.
A single DequantizeLinear converts them back to float at runtime, right before the matmul/conv.
Activations remain in float — no Q node is needed for them.

So the graph doesn't look like → Q → DQ → Op →. Instead it looks like:
INT4 weight (constant) → DQ → MatMul(activation, dequantized_weight)
There's no Q node because the quantization happened statically when the model was exported — the weights are already stored as INT4. You only need DQ to convert them back to float for computation. The single DQ: 1 you see likely means only one layer was eligible (or the tool only quantized one layer of ResNet18 to INT4).

ResNet18 has ~20 Conv layers. Each gets one Q/DQ for weights and one for activations, giving ~40 pairs. You get 38 instead of 40 because a few layers are typically excluded from quantization (e.g., the first conv or final FC layer, which are sensitive to precision loss).

Compare: int8 gives Q:41/DQ:41 — the small difference just reflects which layers each mode's calibration decided to quantize.

In [11]:
ops = [n.op_type for n in onnx.load(f"../onnx/resnet18_int8_qdq.onnx").graph.node]
print(f"Q: {ops.count('QuantizeLinear')}, DQ: {ops.count('DequantizeLinear')}")

Q: 41, DQ: 41


INT4 — 0 Q, 1 DQ (weight-only quantization)

Weights are pre-quantized to INT4 statically at export time and stored as INT4 constants in the graph. No Q node is needed at runtime — the quantization already happened. Only a DQ is inserted to convert INT4 weights → float just before the compute op. Activations stay in float entirely. The "1 DQ" means only 1 layer was eligible (INT4 is too coarse for most of ResNet18's small layers).

INT8 — 41 Q, 41 DQ (full activation + weight quantization)

Both weights and activations are quantized dynamically. Every quantizable op gets a Q/DQ pair on inputs: ~20 Conv layers × 2 tensors (weight + activation) ≈ 40, plus the final FC layer = 41.

FP8 — 38 Q, 38 DQ (3 fewer than INT8)

Same scheme as INT8 (dynamic Q/DQ for both weights and activations), but 3 fewer pairs. This is because modelopt's FP8 mode excludes certain layers that INT8 quantizes:

The first Conv layer is commonly excluded — input pixel distributions don't fit FP8's narrow range well
Residual addition ops or other element-wise ops may be excluded since FP8 arithmetic isn't supported for them in TensorRT
The final FC/classifier may also be excluded for accuracy reasons
INT8 is more permissive about which layers it wraps; FP8 is stricter because the format (E4M3 or E5M2) has a much smaller representable range than INT8.

In [13]:
import onnx
from onnx import TensorProto

model = onnx.load("/home/pf4636/code/resnet/quantized_resnets/onnx/resnet18_fp8_qdq.onnx")

dtype_map = {v: k for k, v in TensorProto.DataType.items()}

for init in model.graph.initializer:
    dtype = dtype_map.get(init.data_type, str(init.data_type))
    print(f"{init.name[:60]:<60} {dtype}")

layer1.0.conv1.weight_zero_point_1                           FLOAT8E4M3FN
layer1.0.conv2.weight_zero_point_1                           FLOAT8E4M3FN
layer1.1.conv1.weight_zero_point_1                           FLOAT8E4M3FN
layer1.1.conv2.weight_zero_point_1                           FLOAT8E4M3FN
layer2.0.conv1.weight_zero_point_1                           FLOAT8E4M3FN
layer2.0.conv2.weight_zero_point_1                           FLOAT8E4M3FN
layer2.0.downsample.0.weight_zero_point_1                    FLOAT8E4M3FN
layer2.1.conv1.weight_zero_point_1                           FLOAT8E4M3FN
layer2.1.conv2.weight_zero_point_1                           FLOAT8E4M3FN
layer3.0.conv1.weight_zero_point_1                           FLOAT8E4M3FN
layer3.0.conv2.weight_zero_point_1                           FLOAT8E4M3FN
layer3.0.downsample.0.weight_zero_point_1                    FLOAT8E4M3FN
layer3.1.conv1.weight_zero_point_1                           FLOAT8E4M3FN
layer3.1.conv2.weight_zero_point_1    